# 🌳 Árboles de Decisión | Clase 4

**Unidad 2:** Métodos Discriminativos
**Duración:** 2 horas
**Versión:** 2025-1 | Modificado: 2026-04-19

## 📋 Mapa de la clase

| Sección | Tema | Tiempo |
|---------|------|--------|
| 1 | Motivación: del modelo paramétrico al no paramétrico | 15 min |
| 2 | Criterios de división: Gini e Impureza de Entropía | 20 min |
| 3 | El algoritmo CART | 20 min |
| 4 | Frontera de decisión y overfitting | 20 min |
| 5 | Importancia de Features | 15 min |
| 6 | Experimento: efecto de max_depth | 20 min |
| 7 | Ejercicio en clase | 10 min |
| 8 | Resumen y conexiones | 10 min |

## 📚 Prerequisitos

### 🔵 Pregrado
- Conceptos de clasificación binaria y multiclase
- Matriz de confusión y métricas (accuracy, precision, recall)
- Noción de overfitting vs underfitting
- Estadística descriptiva básica (probabilidad, media, desviación estándar)

### 🟡 Doctorado
- Todo lo anterior, además:
- Teoría de la información (entropía de Shannon)
- Derivadas parciales y optimización
- Análisis de complejidad algorítmica (notación O)
- Bias-variance decomposition

## 🎯 Objetivos de aprendizaje

Al terminar esta clase podrás:
- Entender cómo funcionan los árboles de decisión como particionadores recursivos del espacio de features
- Calcular Gini e Impureza de Entropía para elegir divisiones óptimas
- Entrenar árboles de decisión con scikit-learn y visualizar su estructura
- Identificar y mitigar overfitting mediante max_depth, min_samples_split y validación cruzada
- Interpretar la importancia de features y explicar decisiones del modelo
- (Doctorado) Derivar formalmente la ganancia de información y comprender CART como algoritmo greedy

In [ ]:
# ── SETUP — NO MODIFICAR ──

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Fijo la semilla para reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Configuración visual
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Carga del dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

print(f"✅ Setup completo")
print(f"   Dataset: Breast Cancer Classification")
print(f"   Muestras: {X.shape[0]}, Features: {X.shape[1]}")
print(f"   Clases: {np.unique(y)} (0=Maligno, 1=Benigno)")
print(f"   Distribución: {np.bincount(y)}")
print(f"\n   NumPy {np.__version__}")
print(f"   Pandas {pd.__version__}")
print(f"   Scikit-learn {__import__('sklearn').__version__}")

# ━━━ SECCIÓN 1: MOTIVACIÓN ━━━

## Motivación: del modelo paramétrico al no paramétrico

En la clase anterior estudiamos clasificadores **probabilísticos generativos** (Naive Bayes, LDA, QDA) que asumen que los datos siguen una **distribución Gaussiana**. Pero, ¿y si los datos NO son gaussianos? ¿Y si la frontera de decisión es muy irregular?

Los **árboles de decisión** son modelos **discriminativos no paramétricos** que no asumen nada sobre la distribución de los datos. En su lugar, **particionan recursivamente el espacio de features** mediante preguntas binarias simples.

### Ejemplo intuitivo: diagnóstico médico

Imagina que eres un médico y debes decidir si un paciente tiene COVID-19:

```
¿Fiebre > 38°C?
├─ NO  → Probablemente no es COVID
└─ SÍ  → Siguiente pregunta
        ¿Tos seca?
        ├─ NO  → Probablemente no es COVID
        └─ SÍ  → Siguiente pregunta
                ¿Pérdida de olfato?
                ├─ NO  → Posible COVID, 60% confianza
                └─ SÍ  → Muy probable COVID, 95% confianza
```

Esta es exactamente la lógica de un **árbol de decisión**: una serie de preguntas que refinan gradualmente la predicción.

### Contraste con modelos anteriores

| Aspecto | NB/LDA/QDA | Árbol de Decisión |
|---------|-----------|-------------------|
| **Forma** | Paramétrico (gaussiano) | No paramétrico |
| **Frontera** | Lineal (LDA) o cuadrática (QDA) | Escalonada (recta paralela a ejes) |
| **Asunciones** | Gaussiana, features independientes (NB) | Ninguna |
| **Interpretabilidad** | Media (necesitas entender parámetros) | **Muy alta** (puedes ver cada decisión) |
| **Riesgo overfitting** | Bajo | **Alto** (sin restricciones) |

# ━━━ SECCIÓN 2: CRITERIOS DE DIVISIÓN ━━━

## Criterios de división: Gini e Impureza de Entropía

La pregunta central en los árboles de decisión es: **¿por dónde divido?**

Para responderla necesitamos una métrica que mida **qué tan "pura" o "homogénea" es un nodo**. Si todos los puntos en un nodo son de la clase 1, el nodo es puro. Si están 50-50 entre dos clases, es impuro.

### Índice de Gini

El **Índice de Gini** mide la probabilidad de clasificar incorrectamente un punto elegido al azar si ignoramos la verdadera etiqueta:

$$\text{Gini}(p) = 1 - \sum_{k=1}^{K} p_k^2$$

donde $p_k$ es la proporción de muestras de la clase $k$ en el nodo.

**Ejemplo numérico:**
- Nodo con 10 muestras: 8 clase 0, 2 clase 1
  - $p_0 = 0.8, p_1 = 0.2$
  - $\text{Gini} = 1 - (0.8^2 + 0.2^2) = 1 - (0.64 + 0.04) = 0.32$

- Nodo puro (10 muestras, todas clase 0):
  - $p_0 = 1.0, p_1 = 0.0$
  - $\text{Gini} = 1 - (1.0^2 + 0.0^2) = 0$

### Impureza de Entropía

La **Entropía** mide la incertidumbre promedio en bits (viene de la teoría de la información de Shannon):

$$\text{Entropía}(p) = -\sum_{k=1}^{K} p_k \log_2(p_k)$$

**Ejemplo numérico (misma distribución 8-2):**
- $\text{Entropía} = -(0.8 \log_2(0.8) + 0.2 \log_2(0.2))$
- $= -(0.8 \times (-0.322) + 0.2 \times (-2.322)) = 0.722$ bits

**Gini vs Entropía:**
- Gini es más simple computacionalmente
- Entropía tiene interpretación teórica (bits de información)
- Para clasificación binaria, casi siempre dan resultados muy similares

### Ganancia de Información

Al dividir un nodo en dos hijos (izquierdo y derecho), queremos maximizar la **reducción de impureza**:

$$\text{Ganancia} = \text{Impureza(padre)} - \left( \frac{n_L}{n} \text{Impureza(izq)} + \frac{n_R}{n} \text{Impureza(der)} \right)$$

donde $n_L$ y $n_R$ son el número de muestras en cada hijo.

In [ ]:
# Ejemplo numérico: Cálculo de Gini y Entropía

def calcular_gini(proporcion_clase):
    """Calcula el índice Gini"""
    return 1 - np.sum(proporcion_clase ** 2)

def calcular_entropia(proporcion_clase):
    """Calcula la entropía en base 2"""
    return -np.sum(proporcion_clase[proporcion_clase > 0] * np.log2(proporcion_clase[proporcion_clase > 0]))

# Nodo con 10 muestras: 8 clase 0, 2 clase 1
prop = np.array([0.8, 0.2])
gini = calcular_gini(prop)
entropia = calcular_entropia(prop)

print("Nodo impuro: 8 muestras clase 0, 2 muestras clase 1")
print(f"  Gini = {gini:.4f}")
print(f"  Entropía = {entropia:.4f} bits")

# Nodo puro: 10 muestras, todas clase 0
prop_puro = np.array([1.0, 0.0])
gini_puro = calcular_gini(prop_puro)
entropia_puro = calcular_entropia(prop_puro)

print("\nNodo puro: 10 muestras clase 0, 0 muestras clase 1")
print(f"  Gini = {gini_puro:.4f}")
print(f"  Entropía = {entropia_puro:.4f} bits")

# Nodo 50-50
prop_50 = np.array([0.5, 0.5])
gini_50 = calcular_gini(prop_50)
entropia_50 = calcular_entropia(prop_50)

print("\nNodo 50-50: 5 muestras clase 0, 5 muestras clase 1")
print(f"  Gini = {gini_50:.4f}")
print(f"  Entropía = {entropia_50:.4f} bits")

> 💡 **Interpretación:**
> - Gini=0 y Entropía=0 significan que el nodo es **puro** (todas las muestras de una clase)
> - Gini=0.5 y Entropía=1 (en binario) significan que el nodo es **máximamente impuro** (50-50)
> - Al dividir, queremos maximizar la ganancia: reducir impureza de padres a hijos
> - En clasificación binaria, Gini y Entropía casi siempre dan las mismas divisiones (correlación > 0.99)

In [ ]:
# Visualización: Gini vs Entropía en función de p

p = np.linspace(0, 1, 100)
gini_vals = 1 - (p**2 + (1-p)**2)
entropia_vals = -p * np.log2(p + 1e-10) - (1-p) * np.log2(1-p + 1e-10)

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(p, gini_vals, label='Gini', linewidth=2, color='#2E86C1')
ax.plot(p, entropia_vals, label='Entropía (base 2)', linewidth=2, color='#D4AC0D')
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='Máxima impureza')
ax.set_xlabel('Proporción de clase positiva (p)', fontsize=11)
ax.set_ylabel('Impureza', fontsize=11)
ax.set_title('Comparación: Índice de Gini vs Entropía', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Nota: Para clasificación binaria, Gini ≈ 0.5 × Entropía")

# ━━━ SECCIÓN 3: EL ALGORITMO CART ━━━

## El algoritmo CART (Classification And Regression Trees)

CART es el algoritmo más usado para construir árboles de decisión. Fue inventado por Breiman et al. (1984) y es lo que usa scikit-learn bajo el capó.

### Algoritmo paso a paso

**Entrada:** Dataset X (n × d), etiquetas y
**Proceso recursivo:**

1. **Inicializa** el nodo con todas las muestras
2. **Si el nodo es puro O alcanza max_depth O tiene < min_samples_split muestras:**
   - DETENTE, crea una hoja con la clase mayoritaria
3. **Si no:**
   - Para cada feature j = 1, ..., d:
     - Para cada umbral t posible (típicamente: cada valor único de Xⱼ):
       - Calcula la ganancia si divides por (j, t)
   - Elige (j*, t*) que maximiza la ganancia
   - Divide recursivamente el sub-árbol izquierdo y derecho

**Salida:** Árbol entrenado con estructura de decisión

### Criterios de parada

Sin restricciones, CART construiría un árbol que memoriza todos los datos (Gini=0). Los parámetros clave para evitar overfitting son:

- **max_depth:** Profundidad máxima del árbol (default: None → sin límite)
- **min_samples_split:** Mínimo de muestras para dividir un nodo (default: 2)
- **min_samples_leaf:** Mínimo de muestras en cada hoja (default: 1)
- **min_impurity_decrease:** Mínima ganancia requerida para dividir (default: 0.0)

### Complejidad computacional

- **Entrenamiento:** O(n · d · log(n)) en el mejor caso (árbol balanceado), O(n · d · n) en el peor caso (árbol degenerado en lista)
- **Predicción:** O(profundidad) ≈ O(log n) para árbol balanceado, O(n) para degenerado
- **Memoria:** O(n) para guardar la estructura del árbol en el peor caso

CART es greedy: en cada nodo elige la división óptima **local**, pero esto NO garantiza el árbol óptimo **global**. Es un trade-off aceptado por eficiencia.

In [ ]:
# Entrenar un árbol de decisión simple

# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE)

# Entrenar árbol con max_depth pequeño para visualizar
tree_simple = DecisionTreeClassifier(
    max_depth=4,
    criterion='gini',
    min_samples_split=20,
    random_state=RANDOM_STATE
)
tree_simple.fit(X_train, y_train)

# Evaluación
y_pred = tree_simple.predict(X_test)
acc_test = accuracy_score(y_test, y_pred)

print(f"Árbol entrenado (max_depth=4):")
print(f"  Accuracy en test: {acc_test:.4f}")
print(f"  Profundidad real: {tree_simple.get_depth()}")
print(f"  Número de nodos: {tree_simple.tree_.node_count}")
print(f"  Número de hojas: {tree_simple.get_n_leaves()}")

In [ ]:
# Visualizar la estructura del árbol

fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(
    tree_simple,
    feature_names=X.columns,
    class_names=['Maligno', 'Benigno'],
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax
)
plt.title('Árbol de Decisión: Clasificación de Cáncer de Mama (max_depth=4)',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("Lectura del árbol:")
print("  - Cada rectángulo es un NODO")
print("  - Arriba: condición de división (ej: 'mean concave points <= 0.05')")
print("  - 'gini': impureza de Gini en ese nodo")
print("  - 'samples': cuántas muestras pasaron por ese nodo")
print("  - 'value': [n_maligno, n_benigno]")
print("  - Color: verde=más benigno, rojo=más maligno")

> 💡 **Lectura de un nodo:**
> - Ejemplo: "mean concave points <= 0.05" con gini=0.495, samples=336, value=[162, 174]
> - Significa: Si el feature "mean concave points" es <= 0.05, ve al hijo izquierdo. Sino, derecha.
> - El nodo tiene 336 muestras: 162 malignas, 174 benignas (casi 50-50, por eso gini alto)
> - Este nodo es **impuro** y se divide más. Una hoja tendría gini=0 (puro).

# ━━━ SECCIÓN 4: FRONTERA DE DECISIÓN Y OVERFITTING ━━━

## Frontera de decisión y overfitting

Una fortaleza de los árboles es que **puedes visualizar exactamente qué frontera aprenden**. A diferencia de LDA/QDA (frontera suave), los árboles crean **fronteras escalonadas** (producto de rectángulos alineados con los ejes).

### El problema de la profundidad

Sin restricciones de profundidad, el árbol puede **memorizar** cada punto de entrenamiento:

- **max_depth muy pequeño:** Underfitting. Frontera demasiado simple, no captura patrones.
- **max_depth muy grande:** Overfitting. Frontera muy compleja, se ajusta al ruido.
- **max_depth óptimo:** Mejor generalización. Frontera lo suficientemente flexible sin memorizar.

Este es el **trade-off bias-varianza:**
- **Bias alto** (underfitting): Modelo simple, predice casi siempre la clase mayoritaria
- **Varianza alta** (overfitting): Pequeños cambios en datos → grandes cambios en árbol

In [ ]:
# Efecto de max_depth en overfitting

# Para visualización, usaremos solo 2 features
feature_ids = [23, 27]  # indices de features
feature_names_2d = ['mean radius', 'worst symmetry']
X_2d = X.iloc[:, feature_ids]

X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d, y, test_size=0.3, random_state=RANDOM_STATE
)

# Entrenar árboles con distintas profundidades
depths = [1, 3, 5, 10, 20]
fig, axes = plt.subplots(1, len(depths), figsize=(18, 4))

for idx, depth in enumerate(depths):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    tree.fit(X_train_2d, y_train_2d)

    # Crear malla para visualizar frontera
    x_min, x_max = X_2d.iloc[:, 0].min() - 1, X_2d.iloc[:, 0].max() + 1
    y_min, y_max = X_2d.iloc[:, 1].min() - 1, X_2d.iloc[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))

    Z = tree.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Graficar
    ax = axes[idx]
    ax.contourf(xx, yy, Z, levels=np.arange(-0.5, 2.5, 1), cmap='RdYlGn', alpha=0.6)
    ax.scatter(X_train_2d.iloc[:, 0], X_train_2d.iloc[:, 1],
              c=y_train_2d, cmap='RdYlGn', edgecolors='black', s=30, alpha=0.7)

    acc_train = accuracy_score(y_train_2d, tree.predict(X_train_2d))
    acc_test = accuracy_score(y_test_2d, tree.predict(X_test_2d))

    ax.set_title(f'Depth={depth}\nTrain:{acc_train:.3f} Test:{acc_test:.3f}', fontsize=10)
    ax.set_xlabel(feature_names_2d[0])
    ax.set_ylabel(feature_names_2d[1])

plt.tight_layout()
plt.show()

print("Observa cómo:")
print("  - depth=1: Frontera muy simple (riesgo: underfitting)")
print("  - depth=3,5: Fronteras complejas pero razonables")
print("  - depth=10,20: Fronteras muy complejas, memorizan ruido (riesgo: overfitting)")

# ━━━ SECCIÓN 5: IMPORTANCIA DE FEATURES ━━━

## Importancia de Features

Una ventaja clave de los árboles es la **interpretabilidad**: puedes saber exactamente qué features son relevantes para la decisión.

### Feature Importance (MDI - Mean Decrease Impurity)

Scikit-learn calcula la importancia de cada feature como la **reducción total de impureza** atribuida a ese feature en todo el árbol:

$$\text{Importancia}(j) = \frac{\sum_{\text{nodos donde divide } j} \text{Ganancia}(j)}{\text{Ganancia total}}$$

Esto responde: "¿Cuánto trabajo hace este feature para purificar los datos?"

### Interpretación

- **Importancia = 0.5:** Este feature es responsable del 50% de la ganancia total
- **Importancia = 0.05:** Este feature contribuye poco; casi no se usa
- Suma de todas las importancias = 1.0

### Limitación: Sesgo hacia features de alta cardinalidad

MDI tiene una limitación importante: **favorece features con muchos valores únicos**. Un feature categórico con 1000 categorías parecerá más importante que uno binario, aunque sea menos informativo.

**Alternativa: Permutation Importance** (sin memorizar, pero en otra sección)

In [ ]:
# Extraer y visualizar feature importance

# Entrenar un árbol más profundo para que capture más detalles
tree_full = DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE, min_samples_split=15)
tree_full.fit(X_train, y_train)

# Obtener importancia
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': tree_full.feature_importances_
}).sort_values('importance', ascending=False)

# Visualizar top 15 features
fig, ax = plt.subplots(figsize=(10, 6))
top_n = 15
importance_top = importance_df.head(top_n)
ax.barh(range(len(importance_top)), importance_top['importance'], color='#2E86C1', alpha=0.8)
ax.set_yticks(range(len(importance_top)))
ax.set_yticklabels(importance_top['feature'], fontsize=10)
ax.set_xlabel('Importancia (reducción de impureza)', fontsize=11)
ax.set_title(f'Top {top_n} Features Más Importantes', fontsize=12, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nTop 10 features:")
print(importance_df.head(10).to_string(index=False))
print(f"\nTotal features con importancia > 0: {(tree_full.feature_importances_ > 0).sum()}")

> 💡 **Interpretación:**
> - El árbol usa principalmente los features "worst..." (caracteríticas peor-caso) para dividir
> - Los features "mean..." contribuyen menos (importancia cercana a 0)
> - Esto tiene sentido: las características extremas (worst) son más discriminativas que los promedios

# ━━━ SECCIÓN 6: EXPERIMENTO — EFECTO DE max_depth ━━━

## Experimento: Efecto de max_depth en el trade-off bias-varianza

Objetivo: Demostrar cómo max_depth controla la curva de overfitting.

**Método:**
1. Barremos max_depth de 1 a 20
2. Para cada profundidad:
   - Entrenamos en X_train, y_train
   - Evaluamos en X_train (error de entrenamiento)
   - Evaluamos en X_test (error de test)
3. Graficamos: accuracy(depth) en train y test

**Esperado:** Curva en U:
- Train: sube (menos underfitting)
- Test: baja, mínimo, sube (overfitting)

In [ ]:
# Barrida de max_depth

depths = range(1, 21)
train_accs = []
test_accs = []
train_f1s = []
test_f1s = []

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE, min_samples_split=10)
    tree.fit(X_train, y_train)

    y_pred_train = tree.predict(X_train)
    y_pred_test = tree.predict(X_test)

    train_accs.append(accuracy_score(y_train, y_pred_train))
    test_accs.append(accuracy_score(y_test, y_pred_test))
    train_f1s.append(f1_score(y_train, y_pred_train))
    test_f1s.append(f1_score(y_test, y_pred_test))

# Encontrar óptimo
best_depth = list(depths)[np.argmax(test_accs)]
best_acc = max(test_accs)

print(f"Mejor max_depth: {best_depth}")
print(f"Mejor accuracy en test: {best_acc:.4f}")

In [ ]:
# Visualizar curva de overfitting

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Accuracy
ax = axes[0]
ax.plot(depths, train_accs, 'o-', label='Train Accuracy', linewidth=2, markersize=6, color='#2E86C1')
ax.plot(depths, test_accs, 's-', label='Test Accuracy', linewidth=2, markersize=6, color='#D4AC0D')
ax.axvline(best_depth, color='red', linestyle='--', linewidth=2, label=f'Óptimo (depth={best_depth})')
ax.set_xlabel('max_depth', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Curva de Overfitting: Accuracy vs Profundidad', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.85, 1.02])

# Gráfico 2: F1-Score
ax = axes[1]
ax.plot(depths, train_f1s, 'o-', label='Train F1', linewidth=2, markersize=6, color='#2E86C1')
ax.plot(depths, test_f1s, 's-', label='Test F1', linewidth=2, markersize=6, color='#D4AC0D')
ax.axvline(best_depth, color='red', linestyle='--', linewidth=2, label=f'Óptimo (depth={best_depth})')
ax.set_xlabel('max_depth', fontsize=11)
ax.set_ylabel('F1-Score', fontsize=11)
ax.set_title('Curva de Overfitting: F1-Score vs Profundidad', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.85, 1.02])

plt.tight_layout()
plt.show()

print("\nObservaciones:")
print(f"  - Train accuracy sube monotónicamente (memoriza)")
print(f"  - Test accuracy sube, alcanza máximo en depth={best_depth}, luego baja")
print(f"  - La brecha train-test explota con profundidad alta: OVERFITTING")

# ━━━ SECCIÓN 7: EJERCICIO EN CLASE ━━━

## Ejercicio Parte 1: Sin computador (10 minutos)

**Enunciado:**
Dado el siguiente conjunto de 8 puntos etiquetados:

| x₁  | x₂  | Clase |
|-----|-----|-------|
| 1.0 | 2.0 | A     |
| 1.5 | 2.5 | A     |
| 2.0 | 1.0 | A     |
| 8.0 | 9.0 | B     |
| 8.5 | 8.5 | B     |
| 9.0 | 9.5 | B     |
| 2.0 | 8.0 | B     |
| 1.0 | 8.5 | B     |

**Preguntas:**

1. Calcula el Gini del nodo raíz (que contiene todos los 8 puntos: 3 clase A, 5 clase B)
   - p_A = 3/8, p_B = 5/8
   - Gini = ?

2. Considera dividir por x₁ ≤ 2.0:
   - **Hijo izquierdo:** puntos con x₁ ≤ 2.0 → {(1.0, 2.0), (1.5, 2.5), (2.0, 1.0), (2.0, 8.0), (1.0, 8.5)} = 3A + 2B
   - **Hijo derecho:** puntos con x₁ > 2.0 → {(8.0, 9.0), (8.5, 8.5), (9.0, 9.5)} = 0A + 3B

   Calcula:
   - Gini(izquierdo) = ?
   - Gini(derecho) = ?
   - Ganancia = ?

3. ¿Cuál es la predicción (clase mayoritaria) en cada hijo?

**Respuestas esperadas:**
```
1. Gini_raíz = 1 - (3/8)² - (5/8)² = 1 - 9/64 - 25/64 = 30/64 ≈ 0.469

2. Izq: Gini = 1 - (3/5)² - (2/5)² = 1 - 9/25 - 4/25 = 12/25 = 0.48
   Der: Gini = 1 - (0/3)² - (3/3)² = 1 - 0 - 1 = 0 (puro!)

   Ganancia = 0.469 - (5/8 × 0.48 + 3/8 × 0) = 0.469 - 0.3 = 0.169 ✓

3. Izq: Clase A (3 vs 2, mayoría A)
   Der: Clase B (0 vs 3, únicamente B)
```

## Ejercicio Parte 2: Con computador (15 minutos)

**Tarea:** Entrena un árbol de decisión en el dataset de breast_cancer y responde:

1. ¿Cuál es el max_depth óptimo según tu experimento anterior?
2. Entrena un árbol con ese max_depth
3. Visualiza el árbol entrenado (usa `plot_tree`)
4. Extrae los 5 features más importantes
5. Predice para una muestra de ejemplo y explica el camino en el árbol

In [ ]:
# [SOLUCIÓN] Ejercicio Parte 2

# 1. Usar el max_depth óptimo del experimento
optimal_depth = best_depth
print(f"1. Max_depth óptimo (del experimento): {optimal_depth}")

# 2. Entrenar árbol
tree_optimal = DecisionTreeClassifier(
    max_depth=optimal_depth,
    random_state=RANDOM_STATE,
    min_samples_split=10
)
tree_optimal.fit(X_train, y_train)

# 3. Evaluación
y_pred = tree_optimal.predict(X_test)
print(f"\n2-3. Árbol entrenado:")
print(f"   Accuracy en test: {accuracy_score(y_test, y_pred):.4f}")
print(f"   Precision: {precision_score(y_test, y_pred):.4f}")
print(f"   Recall: {recall_score(y_test, y_pred):.4f}")
print(f"   F1-Score: {f1_score(y_test, y_pred):.4f}")

# 4. Top 5 features
top_5_features = pd.DataFrame({
    'feature': X.columns,
    'importance': tree_optimal.feature_importances_
}).nlargest(5, 'importance')

print(f"\n4. Top 5 features más importantes:")
for idx, row in top_5_features.iterrows():
    print(f"   {row['feature']}: {row['importance']:.4f}")

In [ ]:
# 5. Predecir para una muestra de ejemplo

# Selecciona una muestra del test set (la primera)
ejemplo_idx = 0
X_ejemplo = X_test.iloc[[ejemplo_idx]]
y_ejemplo_true = y_test.iloc[ejemplo_idx]

# Predicción
y_pred_ejemplo = tree_optimal.predict(X_ejemplo)[0]
y_proba_ejemplo = tree_optimal.predict_proba(X_ejemplo)[0]

print(f"Muestra de ejemplo (índice {ejemplo_idx} en test set):")
print(f"  Features: {X_ejemplo.values[0][:5]}... (primeros 5 de 30)")
print(f"  Etiqueta verdadera: {y_ejemplo_true} ({'Benigno' if y_ejemplo_true == 1 else 'Maligno'})")
print(f"  Predicción del árbol: {y_pred_ejemplo} ({'Benigno' if y_pred_ejemplo == 1 else 'Maligno'})")
print(f"  Confianza: {y_proba_ejemplo[y_pred_ejemplo]:.2%}")
print(f"\n  Probabilidades predichas:")
print(f"    P(Maligno) = {y_proba_ejemplo[0]:.2%}")
print(f"    P(Benigno) = {y_proba_ejemplo[1]:.2%}")
print(f"\n  Resultado: {'✅ CORRECTO' if y_pred_ejemplo == y_ejemplo_true else '❌ INCORRECTO'}")

# ━━━ SECCIÓN 8: RESUMEN Y CONEXIONES ━━━

## Tabla de conceptos clave

| Concepto | Definición | Cuándo usarlo |
|----------|-----------|---------------|
| **Gini** | 1 - Σp_k² | Métrica de impureza (CART estándar, rápido) |
| **Entropía** | -Σp_k log₂(p_k) | Métrica alternativa de impureza (C4.5, teoría info) |
| **Ganancia de Info** | Impureza(padre) - (suma ponderada hijos) | Criterio para elegir divisiones óptimas |
| **CART** | Algoritmo greedy de partición binaria | Construcción de árboles (scikit-learn usa este) |
| **max_depth** | Profundidad máxima del árbol | Controlar overfitting; parámetro de regularización |
| **Feature Importance** | Reducción de impureza por feature | Interpretabilidad: qué features importan |
| **Frontera escalonada** | Rectángulos alineados con ejes | Visualizar decisión del árbol en 2D |
| **Overfitting** | Test accuracy << Train accuracy | Síntoma: max_depth demasiado alto |

## Conexión con la próxima clase: Ensembles

Los árboles individuales tienen un **problema fundamental:** alta varianza. Pequeños cambios en los datos → árbol muy diferente. Esto limita su uso en predicción.

**Solución: Combinar muchos árboles**

La próxima clase cubre dos estrategias:

1. **Random Forests** (Breiman 2001)
   - Entrena B árboles independientes en submuestras bootstrap
   - Promedia predicciones → reduce varianza
   - Paralelizable, escalable

2. **Boosting** (AdaBoost, Gradient Boosting)
   - Entrena árboles secuencialmente, cada uno corrige al anterior
   - Reduce bias principalmente
   - Generalmente más eficaz pero requiere tuning

Ambas logran lo que un árbol solo no puede: predicciones robustas y de baja varianza.

## Resumen de lo aprendido

✅ **Que son los árboles:** Particionadores recursivos no paramétricos del espacio de features
✅ **Cómo se construyen:** CART greedy, maximizando ganancia de información (Gini o Entropía)
✅ **Por qué funcionan:** Flexible (captura no linealidades), interpretable (ves el árbol), discriminativo
✅ **Cuándo falla:** Sin regularización → overfitting; sin features relevantes → basura entra, basura sale
✅ **Cómo regularizar:** max_depth, min_samples_split, validación cruzada
✅ **Cómo interpretar:** Feature importance, visualización del árbol, análisis de path

## Bibliografía

### 🔵 Pregrado
- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly. Cap. 6.
- Müller, A. C., & Guido, S. (2016). *Introduction to Machine Learning with Python*. O'Reilly. Cap. 2.6.
- Scikit-learn oficial: [Decision Trees](https://scikit-learn.org/stable/modules/tree.html)

### 🟡 Doctorado
- **Breiman, L., Friedman, J., Olshen, R., & Stone, C. J. (1984).** *Classification and Regression Trees*. Chapman & Hall. ISBN: 978-0-412-04841-8
  → El paper original de CART. Lectura dura pero fundamental.

- **Hastie, T., Tibshirani, R., & Friedman, J. (2009).** *The Elements of Statistical Learning* (2nd ed.). Springer. Cap. 9 (Tree-Based Methods). doi:10.1007/978-0-387-84858-7
  → Cobertura moderna y rigurosa. Secciones 9.2 (CART) y 9.3 (Regularización)

- **Strobl, C., Boulesteix, A. L., Zeileis, A., & Hothorn, T. (2007).** "Bias in random forest variable importance measures: Illustration, sources and a solution." *BMC Bioinformatics*, 8(25).
  → Crítica a MDI (Mean Decrease Impurity), alternativas. Esencial para interpretabilidad honesta.

- **Quinlan, J. R. (1993).** "C4.5: Programs for machine learning." Morgan Kaufmann.
  → Alternativa histórica a CART (usa Entropía, post-pruning). Buena para comparación.

- **Strobl, C., Hothorn, T., & Zeileis, A. (2009).** "Party on! A new, conditional variable importance measure for random forests available in the party package." *The R Journal*, 1(2).
  → Permutation Importance y Conditional Importance. Más robusto que MDI.

---

**Última actualización:** 2026-04-19
**Versión del curso:** 2025-1